# 51531 — Nilim: Healthcare Policy RAG (LangChain + Semantic Chunking)

**Task:** End-to-end LangChain RAG pipeline over the healthcare policy PDFs with
**semantic chunking**, that answers user questions in a **plain-English, human-readable
summary** instead of dumping raw chunks.

**Pipeline:**

`PDFs → PyPDFLoader → SemanticChunker → AzureOpenAI embeddings → Chroma → retriever → LCEL chain → grounded, summarized answer with citations`


## 1. Setup — Azure OpenAI via LangChain

In [1]:
from pathlib import Path
import os, json, re
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from IPython.display import display, Markdown

from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

ROOT = Path(".")
POLICY_DIR = ROOT / "zs_ai_participants" / "notebook" / "Phase_2" / "data" / "healthcare_policies"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

load_dotenv(".env", override=True)

AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT          = os.getenv("AZURE_OPENAI_MODEL")
EMBEDDING_DEPLOYMENT     = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

for name, val in [
    ("AZURE_OPENAI_API_KEY", AZURE_OPENAI_API_KEY),
    ("AZURE_OPENAI_ENDPOINT", AZURE_OPENAI_ENDPOINT),
    ("AZURE_OPENAI_MODEL", CHAT_DEPLOYMENT),
    ("AZURE_OPENAI_EMBEDDING_MODEL", EMBEDDING_DEPLOYMENT),
]:
    assert val, f"Missing {name} in .env"

embeddings = AzureOpenAIEmbeddings(
    azure_deployment=EMBEDDING_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

llm = AzureChatOpenAI(
    azure_deployment=CHAT_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

display(pd.DataFrame([
    {"Component": "Chat / Generation", "Azure deployment": CHAT_DEPLOYMENT,      "Purpose": "Synthesize the plain-English answer"},
    {"Component": "Embeddings",        "Azure deployment": EMBEDDING_DEPLOYMENT, "Purpose": "Semantic chunking + retrieval"},
]))

C:\Users\nb51531\AppData\Local\Temp\ipykernel_9544\206770768.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\nb51531\AppData\Local\Temp\ipykernel_9544\206770768.py:11: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


,Component,Azure deployment,Purpose
0,Chat / Generation,gpt-4.1,Synthesize the plain-English answer
1,Embeddings,text-embedding-3-small,Semantic chunking + retrieval


## 2. Ingest the healthcare policy PDFs (with governance metadata)

`PyPDFLoader` gives us one `Document` per page. We attach the manifest's
plan/domain metadata so retrieval can be filtered later.

In [2]:
manifest = json.loads((POLICY_DIR / "policy_manifest.json").read_text(encoding="utf-8"))

all_pages = []
for item in manifest:
    pdf_path = POLICY_DIR / item["filename"]
    pages = PyPDFLoader(str(pdf_path)).load()
    for p in pages:
        # Enrich page metadata with the manifest's governance fields.
        p.metadata.update({
            "doc_id":         item["doc_id"],
            "title":          item["title"],
            "plan_type":      item["plan_type"],
            "policy_domain":  item["policy_domain"],
            "effective_date": item["effective_date"],
        })
    all_pages.extend(pages)

doc_df = pd.DataFrame([
    {
        "Document ID": p.metadata["doc_id"],
        "Plan":        p.metadata["plan_type"],
        "Domain":      p.metadata["policy_domain"],
        # PyPDFLoader page metadata is 0-indexed; display as 1-indexed for humans.
        "Page":        p.metadata.get("page", 0) + 1,
        "Chars":       len(p.page_content),
        "Preview":     re.sub(r"\s+", " ", p.page_content)[:110] + "...",
    }
    for p in all_pages
])
display(doc_df)

,Document ID,Plan,Domain,Page,Chars,Preview
0,GOLD-PPO-2026,Gold PPO,Benefits & Authorization,1,1918,Synthetic training document - no real member d...
1,SILVER-HMO-2026,Silver HMO,Benefits & Authorization,1,1560,Synthetic training document - no real member d...
2,IMG-UM-2026,All Commercial Plans,Utilization Management,1,1712,Synthetic training document - no real member d...
3,REHAB-2026,Multiple Plans,Rehabilitation,1,1328,Synthetic training document - no real member d...
4,CLAIMS-OPS-2026,All Commercial Plans,Claims Operations,1,1696,Synthetic training document - no real member d...


## 3. Semantic chunking

`SemanticChunker` embeds each sentence and cuts a new chunk when the semantic
distance to the next sentence spikes past a percentile threshold — so chunks
follow the *meaning* rather than a fixed character window.

This makes an API call per document (sentence embeddings), so it costs a few
extra embedding requests up front, but chunks stay topically coherent.

In [3]:
splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",  # cut where semantic distance > 95th pct
    breakpoint_threshold_amount=95,
)

semantic_chunks = splitter.split_documents(all_pages)

# Give each chunk a stable id for the vector store.
for i, chunk in enumerate(semantic_chunks):
    chunk.metadata["chunk_id"] = f"{chunk.metadata['doc_id']}-{i:03d}"

chunk_stats = (
    pd.DataFrame([
        {"Document": c.metadata["doc_id"], "Chars": len(c.page_content)}
        for c in semantic_chunks
    ])
    .groupby("Document")
    .agg(Chunks=("Chars", "count"), Avg_Chars=("Chars", "mean"), Max_Chars=("Chars", "max"))
    .round(1)
    .reset_index()
)
display(chunk_stats)
print(f"Total semantic chunks: {len(semantic_chunks)}")

,Document,Chunks,Avg_Chars,Max_Chars
0,CLAIMS-OPS-2026,2,847.5,1225
1,GOLD-PPO-2026,2,958.5,1727
2,IMG-UM-2026,2,855.5,1092
3,REHAB-2026,2,663.5,926
4,SILVER-HMO-2026,2,779.5,1118


Total semantic chunks: 10


## 4. Build the Chroma vector store

`Chroma.from_documents` embeds every chunk and stores vectors + metadata in one
call. We use cosine distance so a search score of ~1 means "very similar".

In [4]:
VECTOR_DB_PATH = str(ARTIFACT_DIR / "chroma_51531_nilim")

vectorstore = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embeddings,
    collection_name="policy_semantic_51531",
    persist_directory=VECTOR_DB_PATH,
    collection_metadata={"hnsw:space": "cosine"},
    ids=[c.metadata["chunk_id"] for c in semantic_chunks],
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

display(pd.DataFrame([{
    "Collection":       "policy_semantic_51531",
    "Vectors Stored":   vectorstore._collection.count(),
    "Distance Metric":  "cosine",
    "Retriever Top-K":  4,
}]))

,Collection,Vectors Stored,Distance Metric,Retriever Top-K
0,policy_semantic_51531,10,cosine,4


## 5. Build the RAG chain — grounded, human-readable answer

The prompt does the heavy lifting: it forces a *plain-English summary* rather
than a list of raw chunks, and requires citations only for excerpts actually
used. If evidence is missing, the model must say so — no hallucination.

In [5]:
SYSTEM_PROMPT = """You are a helpful healthcare benefits assistant answering questions
about payer plan policies. You will be given POLICY EXCERPTS and a QUESTION.

Rules:
- Use ONLY the excerpts. Do not use outside knowledge.
- If the excerpts do not contain enough evidence, respond exactly with:
      Answer: I couldn\'t find that in the provided policies.
      Sources: (none)
- Otherwise, write a natural-language answer (2-4 sentences) that a member or
  claims analyst could act on immediately. Rewrite policy language in plain
  English. Do NOT return a bullet list of chunks.
- After the answer, list the sources you actually used.

Output format (exactly):
Answer: <plain-English summary in 2-4 sentences>
Sources: <comma-separated "DOC_ID (page N)" for excerpts you actually used>
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",  "POLICY EXCERPTS:\n{context}\n\nQUESTION: {question}"),
])

def format_context(docs):
    """Render retrieved chunks with inline citations the LLM can echo back."""
    parts = []
    for d in docs:
        doc_id = d.metadata["doc_id"]
        page = d.metadata.get("page", 0) + 1
        parts.append(f"[{doc_id} (page {page})]\n{d.page_content}")
    return "\n\n".join(parts)

rag_chain = (
    {"context": retriever | format_context, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 6. Ask the assistant real questions

In [6]:
def ask(question: str) -> None:
    """Run the RAG chain and render the answer as Markdown."""
    answer = rag_chain.invoke(question)
    display(Markdown(f"### Q: {question}\n\n{answer}"))

questions = [
    "How many chiropractic visits are covered under the Silver HMO plan?",
    "For a Gold PPO member, at what point does physical therapy start requiring prior authorization?",
    "How long does a provider have to file an appeal after a claim is denied?",
    "Is an emergency MRI subject to prior authorization under commercial plans?",
    "Does prior authorization guarantee that a claim will be paid?",
]

for q in questions:
    ask(q)

### Q: How many chiropractic visits are covered under the Silver HMO plan?

Answer: The Silver HMO plan covers up to 12 chiropractic visits per benefit year when they are medically necessary.
Sources: SILVER-HMO-2026 (page 1)

### Q: For a Gold PPO member, at what point does physical therapy start requiring prior authorization?

Answer: For a Gold PPO member, prior authorization for physical therapy is required starting with the 11th visit in a benefit year. The first 10 visits do not require prior authorization.
Sources: GOLD-PPO-2026 (page 1), REHAB-2026 (page 1)

### Q: How long does a provider have to file an appeal after a claim is denied?

Answer: A provider has 60 calendar days from the date of the denial notice to file an appeal, unless a contract or applicable regulation specifies a different timeframe.
Sources: CLAIMS-OPS-2026 (page 1)

### Q: Is an emergency MRI subject to prior authorization under commercial plans?

Answer: Emergency MRI procedures performed as part of an emergency department encounter are not subject to prior authorization under commercial plans. This exemption applies regardless of whether the plan is HMO or PPO.
Sources: SILVER-HMO-2026 (page 1), GOLD-PPO-2026 (page 1), IMG-UM-2026 (page 1)

### Q: Does prior authorization guarantee that a claim will be paid?

Answer: Prior authorization does not guarantee that a claim will be paid. Even if authorization or referral approval is obtained, eligibility, network status, benefit limits, and coding requirements are still evaluated when the claim is processed.
Sources: SILVER-HMO-2026 (page 1), GOLD-PPO-2026 (page 1)

## 7. Peek under the hood — the retrieved evidence

The end-user only sees the summary above, but for governance we can always
show the exact chunks the retriever selected. This is what separates a
grounded system from a black-box generator.

In [7]:
audit_question = "For a Gold PPO member, at what point does physical therapy start requiring prior authorization?"
retrieved = retriever.invoke(audit_question)

display(Markdown(f"**Evidence retrieved for:** _{audit_question}_"))
audit_rows = []
for i, d in enumerate(retrieved):
    audit_rows.append({
        "Rank":     i + 1,
        "Document": d.metadata["doc_id"],
        "Plan":     d.metadata["plan_type"],
        "Page":     d.metadata.get("page", 0) + 1,
        "Chars":    len(d.page_content),
        "Excerpt":  re.sub(r"\s+", " ", d.page_content)[:220] + "...",
    })
display(pd.DataFrame(audit_rows))

**Evidence retrieved for:** _For a Gold PPO member, at what point does physical therapy start requiring prior authorization?_

,Rank,Document,Plan,Page,Chars,Excerpt
0,1,REHAB-2026,Multiple Plans,1,926,SECTION 2: PLAN-SPECIFIC PHYSICAL THERAPY THRE...
1,2,GOLD-PPO-2026,Gold PPO,1,1727,Synthetic training document - no real member d...
2,3,SILVER-HMO-2026,Silver HMO,1,1118,SECTION 2: ADVANCED DIAGNOSTIC IMAGING Non-eme...
3,4,IMG-UM-2026,All Commercial Plans,1,1092,The request must be approved before the schedu...


## Wrap-up

**What this notebook demonstrates:**

1. PDFs → LangChain `PyPDFLoader` with governance metadata attached per page
2. **Semantic chunking** via `SemanticChunker` (cuts at semantic distance peaks)
3. Embed + persist with `AzureOpenAIEmbeddings` + `Chroma`
4. **LCEL RAG chain** that returns a plain-English summary with citations, not raw chunks
5. Auditability: the same retriever can be inspected to see which excerpts were used

The vector store is persisted at `artifacts/chroma_51531_nilim/` so it survives kernel restarts.
